<a href="https://colab.research.google.com/github/MubeenAmjad205/Al-Reverse-Engineer.ipynb/blob/main/AI_Reverse_Engineer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai openai requests beautifulsoup4 markdownify tiktoken tenacity python-dotenv tqdm

In [ ]:
import os
import re
import json
import time
import hashlib
import zipfile
import textwrap
import requests
import tiktoken

from pathlib import Path
from urllib.parse import urlparse
from typing import List, Dict, Any, Tuple
from tqdm.auto import tqdm
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
from google import genai
from openai import OpenAI
from google.genai import types
from google.colab import files
from google.colab import userdata
from bs4 import BeautifulSoup
from markdownify import markdownify as md



In [ ]:
# ============================================================
# API KEYS
# ============================================================

# TINYFISH_API_KEY = "PASTE_YOUR_TINYFISH_API_KEY_HERE"
# GEMINI_API_KEY = "PASTE_YOUR_GEMINI_API_KEY_HERE"

TINYFISH_API_KEY = userdata.get("TINYFISH_API_KEY")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

LLM_PROVIDER = "gemini"  # Options: "gemini", "openai"
# ============================================================
# TARGET WEBSITE CONFIG
# ============================================================

# TARGET_DOMAIN = "example.com"
# ROOT_URL = "https://example.com"
TARGET_DOMAIN = "otstaxi.com"
ROOT_URL = "https://otstaxi.com"

SEARCH_QUERY = f"site:{TARGET_DOMAIN} workflow API documentation features"

MAX_SEARCH_PAGES = 2
MAX_URLS_TO_FETCH = 25

# TinyFish Fetch supports: "markdown", "html", "json"
FETCH_FORMAT = "markdown"

# TinyFish Fetch supports up to 10 URLs per request.
FETCH_BATCH_SIZE = 10

# Rate-limit friendliness.
SLEEP_BETWEEN_SEARCH_CALLS = 1
SLEEP_BETWEEN_FETCH_BATCHES = 2
SLEEP_BETWEEN_GEMINI_CALLS = 1

# ============================================================
# GEMINI MODEL CONFIG
# ============================================================

# Good default for cost-effective analysis.
GEMINI_GEMINI_ANALYSIS_MODEL = "gemini-2.5-flash"
GEMINI_FINAL_REPORT_MODEL = "gemini-2.5-pro"

OPENAI_ANALYSIS_MODEL = "gpt-4o-mini"
OPENAI_FINAL_REPORT_MODEL = "gpt-4o"

TEMPERATURE = 0.2

# ============================================================
# CHUNK CONFIG
# ============================================================

CHUNK_MAX_CHARS = 12000
CHUNK_OVERLAP_CHARS = 800

# ============================================================
# OUTPUT CONFIG
# ============================================================

PROJECT_NAME = "reverse_engineering_docs"

OUTPUT_DIR = Path("/content/reverse_engineering_outputs")
RAW_DIR = OUTPUT_DIR / "raw"
CLEAN_DIR = OUTPUT_DIR / "clean"
CHUNKS_DIR = OUTPUT_DIR / "chunks"
FINDINGS_DIR = OUTPUT_DIR / "findings"

for folder in [OUTPUT_DIR, RAW_DIR, CLEAN_DIR, CHUNKS_DIR, FINDINGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print("Output directory:", OUTPUT_DIR)

Configuration loaded.
Output directory: /content/reverse_engineering_outputs


In [ ]:
# ============================================================
# API KEYS
# ============================================================

# TINYFISH_API_KEY = "PASTE_YOUR_TINYFISH_API_KEY_HERE"
# GEMINI_API_KEY = "PASTE_YOUR_GEMINI_API_KEY_HERE"

TINYFISH_API_KEY = userdata.get("TINYFISH_API_KEY")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

LLM_PROVIDER = "gemini"  # Options: "gemini", "openai"
# ============================================================
# TARGET WEBSITE CONFIG
# ============================================================

# TARGET_DOMAIN = "example.com"
# ROOT_URL = "https://example.com"
TARGET_DOMAIN = "otstaxi.com"
ROOT_URL = "https://otstaxi.com"

SEARCH_QUERY = f"site:{TARGET_DOMAIN} workflow API documentation features"

MAX_SEARCH_PAGES = 2
MAX_URLS_TO_FETCH = 25

# TinyFish Fetch supports: "markdown", "html", "json"
FETCH_FORMAT = "markdown"

# TinyFish Fetch supports up to 10 URLs per request.
FETCH_BATCH_SIZE = 10

# Rate-limit friendliness.
SLEEP_BETWEEN_SEARCH_CALLS = 1
SLEEP_BETWEEN_FETCH_BATCHES = 2
SLEEP_BETWEEN_GEMINI_CALLS = 1

# ============================================================
# GEMINI MODEL CONFIG
# ============================================================

# Good default for cost-effective analysis.
GEMINI_ANALYSIS_MODEL = "gemini-2.5-flash"

# Better for final report quality. You may also use "gemini-2.5-flash"
# if you want lower cost.
GEMINI_FINAL_REPORT_MODEL = "gemini-2.5-flash"

OPENAI_ANALYSIS_MODEL = "gpt-4o-mini"
OPENAI_FINAL_REPORT_MODEL = "gpt-4o"

TEMPERATURE = 0.2

# ============================================================
# CHUNK CONFIG
# ============================================================

CHUNK_MAX_CHARS = 12000
CHUNK_OVERLAP_CHARS = 800

# ============================================================
# OUTPUT CONFIG
# ============================================================

PROJECT_NAME = "reverse_engineering_docs"

OUTPUT_DIR = Path("/content/reverse_engineering_outputs")
RAW_DIR = OUTPUT_DIR / "raw"
CLEAN_DIR = OUTPUT_DIR / "clean"
CHUNKS_DIR = OUTPUT_DIR / "chunks"
FINDINGS_DIR = OUTPUT_DIR / "findings"

for folder in [OUTPUT_DIR, RAW_DIR, CLEAN_DIR, CHUNKS_DIR, FINDINGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print("Output directory:", OUTPUT_DIR)

Configuration loaded.
Output directory: /content/reverse_engineering_outputs


In [ ]:
def validate_config():
    if not TINYFISH_API_KEY or TINYFISH_API_KEY == "PASTE_YOUR_TINYFISH_API_KEY_HERE":
        raise ValueError("Please set TINYFISH_API_KEY.")

    if LLM_PROVIDER == "gemini":
        if not GEMINI_API_KEY or GEMINI_API_KEY == "PASTE_YOUR_GEMINI_API_KEY_HERE":
            raise ValueError("Please set GEMINI_API_KEY when using Gemini.")
    elif LLM_PROVIDER == "openai":
        if not OPENAI_API_KEY or OPENAI_API_KEY == "PASTE_YOUR_OPENAI_API_KEY_HERE":
            raise ValueError("Please set OPENAI_API_KEY when using OpenAI.")
    else:
        raise ValueError("LLM_PROVIDER must be 'gemini' or 'openai'.")

    if not TARGET_DOMAIN:
        raise ValueError("Please set TARGET_DOMAIN.")

    if not ROOT_URL.startswith(("http://", "https://")):
        raise ValueError("ROOT_URL must start with http:// or https://")


validate_config()

gemini_client = None
openai_client = None

if LLM_PROVIDER == "gemini":
    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    print("TinyFish and Gemini configuration validated.")
elif LLM_PROVIDER == "openai":
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    print("TinyFish and OpenAI configuration validated.")



TinyFish and Gemini configuration validated.


In [ ]:
def safe_filename(value: str, max_len: int = 120) -> str:
    value = re.sub(r"https?://", "", value)
    value = re.sub(r"[^a-zA-Z0-9._-]+", "_", value)
    value = value.strip("_")

    if len(value) > max_len:
        digest = hashlib.sha256(value.encode("utf-8")).hexdigest()[:10]
        value = value[:max_len - 11] + "_" + digest

    return value or "untitled"


def normalize_url(url: str) -> str:
    parsed = urlparse(url)

    scheme = parsed.scheme or "https"
    netloc = parsed.netloc.lower()
    path = parsed.path or "/"

    normalized = f"{scheme}://{netloc}{path}"

    return normalized.rstrip("/")


def is_same_domain(url: str, target_domain: str) -> bool:
    try:
        host = urlparse(url).netloc.lower()
        target = target_domain.lower()
        target = target.replace("https://", "").replace("http://", "").strip("/")

        return host == target or host.endswith("." + target)
    except Exception:
        return False


def remove_duplicate_urls(urls: List[str]) -> List[str]:
    seen = set()
    output = []

    for url in urls:
        if not url:
            continue

        normalized = normalize_url(url)

        if normalized not in seen:
            seen.add(normalized)
            output.append(normalized)

    return output


def save_json(path: Path, data: Any):
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def save_text(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def load_text(path: Path) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def now_timestamp() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")


def batch_list(items: List[Any], batch_size: int) -> List[List[Any]]:
    return [items[i:i + batch_size] for i in range(0, len(items), batch_size)]

In [ ]:
class TinyFishClient:
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.search_endpoint = "https://api.search.tinyfish.ai"
        self.fetch_endpoint = "https://api.fetch.tinyfish.ai"

    @property
    def json_headers(self) -> Dict[str, str]:
        return {
            "X-API-Key": self.api_key,
            "Accept": "application/json",
            "Content-Type": "application/json",
        }

    @property
    def get_headers(self) -> Dict[str, str]:
        return {
            "X-API-Key": self.api_key,
            "Accept": "application/json",
        }

    @retry(
        wait=wait_exponential(multiplier=1, min=2, max=30),
        stop=stop_after_attempt(4),
        retry=retry_if_exception_type(requests.RequestException),
    )
    def search(
        self,
        query: str,
        location: str = "US",
        language: str = "en",
        page: int = 0,
        timeout: int = 60,
    ) -> Dict[str, Any]:
        params = {
            "query": query,
            "location": location,
            "language": language,
            "page": page,
        }

        response = requests.get(
            self.search_endpoint,
            headers=self.get_headers,
            params=params,
            timeout=timeout,
        )

        if response.status_code == 429:
            raise requests.RequestException("TinyFish Search rate limit exceeded.")

        response.raise_for_status()
        return response.json()

    @retry(
        wait=wait_exponential(multiplier=1, min=2, max=30),
        stop=stop_after_attempt(4),
        retry=retry_if_exception_type(requests.RequestException),
    )
    def fetch(
        self,
        urls: List[str],
        output_format: str = "markdown",
        links: bool = True,
        image_links: bool = False,
        timeout: int = 180,
    ) -> Dict[str, Any]:
        if len(urls) > 10:
            raise ValueError("TinyFish Fetch allows maximum 10 URLs per request.")

        payload = {
            "urls": urls,
            "format": output_format,
            "links": links,
            "image_links": image_links,
        }

        response = requests.post(
            self.fetch_endpoint,
            headers=self.json_headers,
            json=payload,
            timeout=timeout,
        )

        if response.status_code == 429:
            raise requests.RequestException("TinyFish Fetch rate limit exceeded.")

        response.raise_for_status()
        return response.json()


tinyfish = TinyFishClient(TINYFISH_API_KEY)

print("TinyFish client ready.")

TinyFish client ready.


In [ ]:
def discover_urls_with_search(
    query: str,
    target_domain: str,
    max_pages: int = 2,
    location: str = "US",
    language: str = "en",
) -> List[str]:
    all_urls = []
    raw_search_results = []

    for page in range(max_pages):
        print(f"Searching page {page}: {query}")

        data = tinyfish.search(
            query=query,
            location=location,
            language=language,
            page=page,
        )

        raw_search_results.append(data)

        results = data.get("results", [])

        for item in results:
            url = item.get("url")
            if url and is_same_domain(url, target_domain):
                all_urls.append(url)

        time.sleep(SLEEP_BETWEEN_SEARCH_CALLS)

    save_json(OUTPUT_DIR / "tinyfish_search_results.json", raw_search_results)

    urls = remove_duplicate_urls(all_urls)
    return urls


search_urls = discover_urls_with_search(
    query=SEARCH_QUERY,
    target_domain=TARGET_DOMAIN,
    max_pages=MAX_SEARCH_PAGES,
)

print(f"Discovered {len(search_urls)} URLs from search.")

for url in search_urls[:20]:
    print("-", url)

Searching page 0: site:otstaxi.com workflow API documentation features
Searching page 1: site:otstaxi.com workflow API documentation features
Discovered 0 URLs from search.


In [ ]:
MANUAL_URLS = [
    ROOT_URL,

    # Add more if needed:
    # "https://example.com/docs",
    # "https://example.com/api",
    # "https://example.com/pricing",
]

all_seed_urls = remove_duplicate_urls(search_urls + MANUAL_URLS)

all_seed_urls = [
    url for url in all_seed_urls
    if is_same_domain(url, TARGET_DOMAIN)
]

all_seed_urls = all_seed_urls[:MAX_URLS_TO_FETCH]

save_json(OUTPUT_DIR / "seed_urls.json", all_seed_urls)

print(f"Total seed URLs: {len(all_seed_urls)}")

for url in all_seed_urls:
    print("-", url)

Total seed URLs: 1
- https://otstaxi.com


In [ ]:
def fetch_pages(urls: List[str]) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    all_results = []
    all_errors = []

    batches = batch_list(urls, FETCH_BATCH_SIZE)

    for batch_index, batch in enumerate(tqdm(batches, desc="Fetching batches"), start=1):
        print(f"\nFetching batch {batch_index}/{len(batches)}")

        for url in batch:
            print(" -", url)

        try:
            data = tinyfish.fetch(
                urls=batch,
                output_format=FETCH_FORMAT,
                links=True,
                image_links=False,
            )

            results = data.get("results", [])
            errors = data.get("errors", [])

            all_results.extend(results)
            all_errors.extend(errors)

            save_json(RAW_DIR / f"fetch_batch_{batch_index:03d}.json", data)

        except Exception as e:
            error_obj = {
                "batch_index": batch_index,
                "urls": batch,
                "error": str(e),
            }

            all_errors.append(error_obj)
            print("Batch failed:", e)

        time.sleep(SLEEP_BETWEEN_FETCH_BATCHES)

    save_json(OUTPUT_DIR / "fetch_results_all.json", all_results)
    save_json(OUTPUT_DIR / "fetch_errors_all.json", all_errors)

    return all_results, all_errors


fetch_results, fetch_errors = fetch_pages(all_seed_urls)

print("\nFetch complete.")
print("Successful pages:", len(fetch_results))
print("Errors:", len(fetch_errors))

if fetch_errors:
    print("\nFirst errors:")
    for error in fetch_errors[:10]:
        print(error)

Fetching batches:   0%|          | 0/1 [00:00<?, ?it/s]


Fetching batch 1/1
 - https://otstaxi.com

Fetch complete.
Successful pages: 1
Errors: 0


In [ ]:
def extract_links_from_fetch_results(
    fetch_results: List[Dict[str, Any]],
    target_domain: str,
    max_links: int = 30,
) -> List[str]:
    links = []

    for page in fetch_results:
        page_links = page.get("links", []) or []

        for link in page_links:
            if is_same_domain(link, target_domain):
                links.append(link)

    links = remove_duplicate_urls(links)

    already = set(normalize_url(u) for u in all_seed_urls)

    links = [
        url for url in links
        if normalize_url(url) not in already
    ]

    return links[:max_links]


EXPAND_LINKS_ONE_LEVEL = True
MAX_EXTRA_LINKS = 20

if EXPAND_LINKS_ONE_LEVEL:
    extra_urls = extract_links_from_fetch_results(
        fetch_results=fetch_results,
        target_domain=TARGET_DOMAIN,
        max_links=MAX_EXTRA_LINKS,
    )

    print(f"Extra URLs discovered from fetched pages: {len(extra_urls)}")

    for url in extra_urls:
        print("-", url)

    if extra_urls:
        extra_results, extra_errors = fetch_pages(extra_urls)

        fetch_results.extend(extra_results)
        fetch_errors.extend(extra_errors)

        save_json(OUTPUT_DIR / "fetch_results_all_expanded.json", fetch_results)
        save_json(OUTPUT_DIR / "fetch_errors_all_expanded.json", fetch_errors)
else:
    print("One-level link expansion disabled.")

Extra URLs discovered from fetched pages: 20
- https://otstaxi.com/contact-us
- https://otstaxi.com/about-us
- https://otstaxi.com/airports
- https://otstaxi.com/partners
- https://otstaxi.com/account/login
- https://otstaxi.com/account/register
- https://otstaxi.com/airport/bhd
- https://otstaxi.com/airport/cwl
- https://otstaxi.com/airport/syy
- https://otstaxi.com/airport/nqy
- https://otstaxi.com/airport/ldy
- https://otstaxi.com/airport/stn
- https://otstaxi.com/airport/nwi
- https://otstaxi.com/airport/sou
- https://otstaxi.com/airport/ema
- https://otstaxi.com/airport/lcy
- https://otstaxi.com/airport/man
- https://otstaxi.com/airport/dnd
- https://otstaxi.com/airport/ext
- https://otstaxi.com/airport/bfs


Fetching batches:   0%|          | 0/2 [00:00<?, ?it/s]


Fetching batch 1/2
 - https://otstaxi.com/contact-us
 - https://otstaxi.com/about-us
 - https://otstaxi.com/airports
 - https://otstaxi.com/partners
 - https://otstaxi.com/account/login
 - https://otstaxi.com/account/register
 - https://otstaxi.com/airport/bhd
 - https://otstaxi.com/airport/cwl
 - https://otstaxi.com/airport/syy
 - https://otstaxi.com/airport/nqy

Fetching batch 2/2
 - https://otstaxi.com/airport/ldy
 - https://otstaxi.com/airport/stn
 - https://otstaxi.com/airport/nwi
 - https://otstaxi.com/airport/sou
 - https://otstaxi.com/airport/ema
 - https://otstaxi.com/airport/lcy
 - https://otstaxi.com/airport/man
 - https://otstaxi.com/airport/dnd
 - https://otstaxi.com/airport/ext
 - https://otstaxi.com/airport/bfs


In [ ]:
BOILERPLATE_PATTERNS = [
    r"(?i)accept all cookies",
    r"(?i)cookie policy",
    r"(?i)privacy policy",
    r"(?i)terms of service",
    r"(?i)subscribe to newsletter",
    r"(?i)all rights reserved",
    r"(?i)sign up",
    r"(?i)log in",
    r"(?i)©",
]


def get_page_text(page: Dict[str, Any]) -> str:
    """
    TinyFish commonly returns extracted page content as 'text'.
    Some wrappers/examples may call it 'content'.
    This handles both safely.
    """
    text = (
        page.get("text")
        or page.get("content")
        or page.get("markdown")
        or ""
    )

    if isinstance(text, dict):
        text = json.dumps(text, indent=2, ensure_ascii=False)

    if isinstance(text, list):
        text = "\n".join(str(x) for x in text)

    return str(text)


def clean_markdown_text(text: str) -> str:
    if not text:
        return ""

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{4,}", "\n\n\n", text)

    lines = text.splitlines()
    cleaned_lines = []

    for line in lines:
        stripped = line.strip()

        if not stripped:
            cleaned_lines.append("")
            continue

        skip = False

        for pattern in BOILERPLATE_PATTERNS:
            if re.search(pattern, stripped) and len(stripped) < 180:
                skip = True
                break

        if skip:
            continue

        cleaned_lines.append(stripped)

    cleaned = "\n".join(cleaned_lines)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)

    return cleaned.strip()


def prepare_clean_documents(fetch_results: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    docs = []

    for page in fetch_results:
        url = page.get("url") or ""
        final_url = page.get("final_url") or page.get("url") or ""
        title = page.get("title") or ""
        description = page.get("description") or ""

        raw_text = get_page_text(page)
        cleaned = clean_markdown_text(raw_text)

        if len(cleaned) < 200:
            continue

        doc = {
            "url": url,
            "final_url": final_url,
            "title": title,
            "description": description,
            "language": page.get("language"),
            "published_date": page.get("published_date"),
            "content": cleaned,
            "content_length": len(cleaned),
        }

        docs.append(doc)

        filename = safe_filename(final_url) + ".md"

        save_text(
            CLEAN_DIR / filename,
            f"# {title or final_url}\n\n"
            f"Source: {final_url}\n\n"
            f"---\n\n"
            f"{cleaned}"
        )

    save_json(OUTPUT_DIR / "clean_documents.json", docs)

    return docs


clean_docs = prepare_clean_documents(fetch_results)

print(f"Prepared {len(clean_docs)} clean documents.")

for doc in clean_docs[:10]:
    print("-", doc["title"][:80], "|", doc["final_url"], "| chars:", doc["content_length"])

Prepared 12 clean documents.
- OTS Taxi | Airport Taxi & Transfers | Online Travel Services | https://otstaxi.com/ | chars: 1340
- Contact OTS Taxi | Airport Taxi & Transfers | Online Travel Services | https://otstaxi.com/contact-us | chars: 641
- About OTS Taxi | https://otstaxi.com/about-us | chars: 2277
- Airports - We support major and regional airports | https://otstaxi.com/airports | chars: 3351
- Cornwall Airport Newquay Taxi & Airport Transfer | https://otstaxi.com/airport/nqy | chars: 7757
- City of Derry Airport Taxi & Airport Transfer | https://otstaxi.com/airport/ldy | chars: 4854
- London Stansted Airport Taxi & Airport Transfer | https://otstaxi.com/airport/stn | chars: 5865
- Southampton Airport Taxi & Airport Transfer | https://otstaxi.com/airport/sou | chars: 8341
- East Midlands Airport Taxi & Airport Transfer | https://otstaxi.com/airport/ema | chars: 8674
- Dundee Airport Taxi & Airport Transfer | https://otstaxi.com/airport/dnd | chars: 5076


In [ ]:
def content_hash(text: str) -> str:
    normalized = re.sub(r"\s+", " ", text.lower()).strip()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


def deduplicate_documents(docs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    unique_docs = []

    for doc in docs:
        digest = content_hash(doc["content"])

        if digest not in seen:
            seen.add(digest)
            unique_docs.append(doc)

    return unique_docs


unique_docs = deduplicate_documents(clean_docs)

save_json(OUTPUT_DIR / "unique_documents.json", unique_docs)

print(f"Unique documents: {len(unique_docs)}")

Unique documents: 12


In [ ]:
def chunk_with_overlap(text: str, max_chars: int, overlap_chars: int) -> List[str]:
    if len(text) <= max_chars:
        return [text]

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + max_chars, len(text))
        window = text[start:end]

        break_points = [
            window.rfind("\n## "),
            window.rfind("\n# "),
            window.rfind("\n\n"),
            window.rfind(". "),
        ]

        break_at = max(break_points)

        if break_at > max_chars * 0.5:
            end = start + break_at

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = max(end - overlap_chars, start + 1)

    return chunks


def build_chunks(docs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    all_chunks = []

    for doc_index, doc in enumerate(docs, start=1):
        doc_chunks = chunk_with_overlap(
            doc["content"],
            max_chars=CHUNK_MAX_CHARS,
            overlap_chars=CHUNK_OVERLAP_CHARS,
        )

        for chunk_index, chunk in enumerate(doc_chunks, start=1):
            chunk_obj = {
                "doc_index": doc_index,
                "chunk_index": chunk_index,
                "chunk_id": f"doc{doc_index:03d}_chunk{chunk_index:03d}",
                "url": doc["url"],
                "final_url": doc["final_url"],
                "title": doc["title"],
                "description": doc["description"],
                "content": chunk,
                "content_length": len(chunk),
            }

            all_chunks.append(chunk_obj)

            save_text(
                CHUNKS_DIR / f"{chunk_obj['chunk_id']}.md",
                f"# Chunk {chunk_obj['chunk_id']}\n\n"
                f"Title: {doc['title']}\n\n"
                f"URL: {doc['final_url']}\n\n"
                f"---\n\n"
                f"{chunk}"
            )

    save_json(OUTPUT_DIR / "chunks.json", all_chunks)

    return all_chunks


chunks = build_chunks(unique_docs)

print(f"Total chunks: {len(chunks)}")

for chunk in chunks[:10]:
    print(chunk["chunk_id"], "| chars:", chunk["content_length"], "|", chunk["final_url"])

Total chunks: 12
doc001_chunk001 | chars: 1340 | https://otstaxi.com/
doc002_chunk001 | chars: 641 | https://otstaxi.com/contact-us
doc003_chunk001 | chars: 2277 | https://otstaxi.com/about-us
doc004_chunk001 | chars: 3351 | https://otstaxi.com/airports
doc005_chunk001 | chars: 7757 | https://otstaxi.com/airport/nqy
doc006_chunk001 | chars: 4854 | https://otstaxi.com/airport/ldy
doc007_chunk001 | chars: 5865 | https://otstaxi.com/airport/stn
doc008_chunk001 | chars: 8341 | https://otstaxi.com/airport/sou
doc009_chunk001 | chars: 8674 | https://otstaxi.com/airport/ema
doc010_chunk001 | chars: 5076 | https://otstaxi.com/airport/dnd


In [ ]:
CHUNK_ANALYSIS_SYSTEM_PROMPT = """
You are a senior reverse engineer and technical analyst.

Your task:
Analyze the provided website/page/code/document content and reverse engineer ONLY the algorithmic behavior.

Do NOT explain:
- syntax
- visual design
- branding
- file structure
- class structure
- implementation quality
unless it is required to understand the actual algorithm.

Focus on:
- execution logic
- user/system workflow
- input processing
- output generation
- decision points
- validation/filtering
- state tracking
- session/task lifecycle
- triggers
- waits
- retries
- failure recovery
- concurrency/parallelism
- queues/workers
- async orchestration
- automation or detection behavior if observable
- DOM interaction behavior if observable
- API/data transformation logic if observable

Important:
If this is public website content and source code is not available, mark findings as INFERRED.
Do not invent backend behavior.
Use "Not observable from provided content" where needed.

Return Markdown only.
"""


CHUNK_ANALYSIS_USER_TEMPLATE = """
Analyze this source chunk.

SOURCE METADATA:
- Title: {title}
- URL: {url}
- Chunk ID: {chunk_id}

CONTENT:
{content}

Return this exact structure:

# Chunk Algorithmic Analysis: {chunk_id}

## Source Context
- URL:
- Title:
- Confidence:
- Confirmed vs inferred notes:

## Main Algorithm
Step-by-step execution flow.

## Input → Processing → Output
Inputs, transformation stages, outputs.

## Decision Tree
Important conditions and branches.

## Data Flow
How data moves through the visible/inferred system.

## State Flow
Tracked state and state changes over time.

## Control Flow
Triggers, waits, retries, exits, failure handling.

## Concurrency Model
Parallelism, queues, workers, async behavior, or not observable.

## Detection / Automation Logic
Timing, synchronization, fingerprinting, session handling, DOM interaction, or not observable.

## Plain English Algorithm
Simple behavioral explanation.

## Pseudocode
Concise pseudocode.

## Flowchart-Style Text
START
-> ...
-> END

## Unknowns / Not Observable
List only meaningful unknowns.
"""


FINAL_REPORT_SYSTEM_PROMPT = """
You are a senior reverse engineer writing a professional Markdown report.

Your job:
Merge many chunk-level algorithmic findings into one coherent reverse-engineering document.

Rules:
- Focus only on actual behavior and execution logic.
- Remove duplicates.
- Separate confirmed behavior from inferred behavior.
- Do not discuss code style.
- Do not invent backend logic.
- Prefer "not observable" over guessing.
- Preserve source URLs where useful.
- Write polished professional English.
- Output Markdown only.
"""


FINAL_REPORT_USER_TEMPLATE = """
Create a final professional reverse-engineering Markdown report.

PROJECT:
{project_name}

TARGET DOMAIN:
{target_domain}

ROOT URL:
{root_url}

ANALYSIS GENERATED AT:
{timestamp}

CHUNK FINDINGS:
{findings}

Required report structure:

# Reverse Engineering Report: {project_name}

## 1. Executive Summary

## 2. Scope

## 3. Source Coverage
Mention what URLs/pages were analyzed.

## 4. Confidence Model
Separate:
- Confirmed behavior
- Inferred behavior
- Not observable

## 5. Main Algorithm
- Step-by-step execution flow
- Input → processing → output
- Decision tree
- Internal logic flow

## 6. Data Flow
- How data moves through the system
- Transformation stages
- Validation/filtering stages

## 7. State Flow
- What state is tracked
- How state changes over time
- Session/task lifecycle

## 8. Control Flow
- What triggers what
- Retry conditions
- Wait conditions
- Exit conditions
- Failure recovery logic

## 9. Concurrency Model
- Parallelism
- Queues
- Workers
- Async orchestration

## 10. Detection / Automation Logic
- Timing strategy
- Synchronization strategy
- Fingerprinting/session handling
- DOM interaction logic

## 11. Plain English Algorithm

## 12. Pseudocode

## 13. Flowchart-Style Execution

Use this style:

START
-> Load task
-> Create session
-> Navigate
IF condition:
  -> Branch
ELSE:
  -> Branch
-> Save result
-> END

## 14. Unknowns / Not Observable

## 15. Final Notes
"""

In [ ]:
def _call_gemini(model: str, system_prompt: str, user_prompt: str, temperature: float) -> str:
    full_prompt = f"""
SYSTEM INSTRUCTIONS:
{system_prompt}

USER TASK:
{user_prompt}
"""
    response = gemini_client.models.generate_content(
        model=model,
        contents=full_prompt,
        config=types.GenerateContentConfig(
            temperature=temperature,
        ),
    )

    text = getattr(response, "text", None)
    if text and text.strip():
        return text.strip()

    try:
        parts = response.candidates[0].content.parts
        joined = "\n".join(
            getattr(part, "text", "")
            for part in parts
            if getattr(part, "text", "")
        ).strip()
        if joined:
            return joined
    except Exception:
        pass

    raise RuntimeError("Gemini returned an empty response.")


def _call_openai(model: str, system_prompt: str, user_prompt: str, temperature: float) -> str:
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=temperature,
    )
    
    text = response.choices[0].message.content
    if text and text.strip():
        return text.strip()
        
    raise RuntimeError("OpenAI returned an empty response.")


def call_llm(
    model: str,
    system_prompt: str,
    user_prompt: str,
    temperature: float = 0.2,
    max_retries: int = 3,
) -> str:
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            if LLM_PROVIDER == "gemini":
                return _call_gemini(model, system_prompt, user_prompt, temperature)
            elif LLM_PROVIDER == "openai":
                return _call_openai(model, system_prompt, user_prompt, temperature)
            else:
                raise ValueError("Invalid LLM_PROVIDER")

        except Exception as e:
            last_error = e
            print(f"{LLM_PROVIDER.capitalize()} call failed on attempt {attempt}/{max_retries}: {e}")
            import time
            time.sleep(2 * attempt)

    raise RuntimeError(f"{LLM_PROVIDER.capitalize()} call failed after {max_retries} attempts: {last_error}")


def estimate_tokens_rough(text: str) -> int:
    try:
        enc = tiktoken.get_encoding("cl100k_base")
        return len(enc.encode(text))
    except Exception:
        return max(1, len(text) // 4)



In [ ]:
def analyze_chunks_with_llm(chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    findings = []

    for chunk in tqdm(chunks, desc="Analyzing chunks with Gemini"):
        chunk_id = chunk["chunk_id"]
        finding_path = FINDINGS_DIR / f"{chunk_id}_analysis.md"

        # Resume support.
        if finding_path.exists():
            analysis = load_text(finding_path)

            findings.append({
                "chunk_id": chunk_id,
                "url": chunk["final_url"],
                "title": chunk["title"],
                "analysis": analysis,
            })

            continue

        user_prompt = CHUNK_ANALYSIS_USER_TEMPLATE.format(
            title=chunk["title"] or "Untitled",
            url=chunk["final_url"],
            chunk_id=chunk_id,
            content=chunk["content"],
        )

        print(f"\nAnalyzing {chunk_id}")
        print("URL:", chunk["final_url"])
        print("Estimated prompt tokens:", estimate_tokens_rough(user_prompt))

        analysis = call_llm(
            model=GEMINI_ANALYSIS_MODEL if LLM_PROVIDER == "gemini" else OPENAI_ANALYSIS_MODEL,
            system_prompt=CHUNK_ANALYSIS_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            temperature=TEMPERATURE,
        )

        save_text(finding_path, analysis)

        findings.append({
            "chunk_id": chunk_id,
            "url": chunk["final_url"],
            "title": chunk["title"],
            "analysis": analysis,
        })

        time.sleep(SLEEP_BETWEEN_GEMINI_CALLS)

    save_json(OUTPUT_DIR / "chunk_findings.json", findings)

    return findings


chunk_findings = analyze_chunks_with_llm(chunks)

print(f"Chunk analyses complete: {len(chunk_findings)}")

Analyzing chunks with Gemini:   0%|          | 0/12 [00:00<?, ?it/s]


Analyzing doc001_chunk001
URL: https://otstaxi.com/
Estimated prompt tokens: 523

Analyzing doc002_chunk001
URL: https://otstaxi.com/contact-us
Estimated prompt tokens: 405

Analyzing doc003_chunk001
URL: https://otstaxi.com/about-us
Estimated prompt tokens: 640

Analyzing doc004_chunk001
URL: https://otstaxi.com/airports
Estimated prompt tokens: 1144

Analyzing doc005_chunk001
URL: https://otstaxi.com/airport/nqy
Estimated prompt tokens: 1765

Analyzing doc006_chunk001
URL: https://otstaxi.com/airport/ldy
Estimated prompt tokens: 1245

Analyzing doc007_chunk001
URL: https://otstaxi.com/airport/stn
Estimated prompt tokens: 1364

Analyzing doc008_chunk001
URL: https://otstaxi.com/airport/sou
Estimated prompt tokens: 1753

Analyzing doc009_chunk001
URL: https://otstaxi.com/airport/ema
Estimated prompt tokens: 1879

Analyzing doc010_chunk001
URL: https://otstaxi.com/airport/dnd
Estimated prompt tokens: 1277

Analyzing doc011_chunk001
URL: https://otstaxi.com/airport/ext
Estimated prompt 

In [ ]:
def build_findings_bundle(findings: List[Dict[str, Any]], max_chars: int = 140000) -> str:
    sections = []

    for item in findings:
        section = f"""
---
CHUNK ID: {item['chunk_id']}
TITLE: {item['title']}
URL: {item['url']}

{item['analysis']}
"""

        sections.append(section)

    bundle = "\n\n".join(sections)

    if len(bundle) > max_chars:
        print(f"Findings bundle is large ({len(bundle)} chars). Trimming to {max_chars} chars.")
        bundle = bundle[:max_chars]

    return bundle


def build_final_report_with_llm(findings: List[Dict[str, Any]]) -> str:
    findings_bundle = build_findings_bundle(findings)

    user_prompt = FINAL_REPORT_USER_TEMPLATE.format(
        project_name=PROJECT_NAME,
        target_domain=TARGET_DOMAIN,
        root_url=ROOT_URL,
        timestamp=now_timestamp(),
        findings=findings_bundle,
    )

    print("Building final report with Gemini...")
    print("Estimated prompt tokens:", estimate_tokens_rough(user_prompt))

    final_report = call_llm(
        model=GEMINI_FINAL_REPORT_MODEL if LLM_PROVIDER == "gemini" else OPENAI_FINAL_REPORT_MODEL,
        system_prompt=FINAL_REPORT_SYSTEM_PROMPT,
        user_prompt=user_prompt,
        temperature=TEMPERATURE,
    )

    return final_report


final_report = build_final_report_with_llm(chunk_findings)

final_report_path = OUTPUT_DIR / "reverse_engineering_report.md"

save_text(final_report_path, final_report)

print("Final report saved to:", final_report_path)
print("\nPreview:")
print(final_report[:3000])

Findings bundle is large (163271 chars). Trimming to 140000 chars.
Building final report with Gemini...
Estimated prompt tokens: 31885
Gemini call failed on attempt 1/3: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, m

RuntimeError: Gemini call failed after 3 attempts: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\nPlease retry in 59.503807359s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.goog

In [ ]:
SUMMARY_SYSTEM_PROMPT = """
You are a senior technical writer.

Create a concise executive summary from a reverse-engineering report.
Use professional English.
Return Markdown only.
"""


def build_short_summary_with_llm(report: str) -> str:
    user_prompt = f"""
Create a short professional summary of this reverse-engineering report.

Include:
- system purpose
- main algorithm
- key data flow
- key control flow
- what is confirmed vs inferred
- major unknowns

REPORT:
{report[:90000]}
"""

    return call_llm(
        model=GEMINI_ANALYSIS_MODEL if LLM_PROVIDER == "gemini" else OPENAI_ANALYSIS_MODEL,
        system_prompt=SUMMARY_SYSTEM_PROMPT,
        user_prompt=user_prompt,
        temperature=0.2,
    )


short_summary = build_short_summary_with_llm(final_report)

summary_path = OUTPUT_DIR / "short_algorithm_summary.md"

save_text(summary_path, short_summary)

print("Short summary saved to:", summary_path)
print(short_summary[:2000])

In [ ]:
def build_source_index(docs: List[Dict[str, Any]]) -> str:
    lines = [
        "# Source Index",
        "",
        f"Generated: {now_timestamp()}",
        "",
        f"Target domain: `{TARGET_DOMAIN}`",
        "",
        "## Analyzed Sources",
        "",
    ]

    for index, doc in enumerate(docs, start=1):
        lines.append(f"### {index}. {doc.get('title') or 'Untitled'}")
        lines.append("")
        lines.append(f"- URL: {doc.get('final_url')}")
        lines.append(f"- Original URL: {doc.get('url')}")
        lines.append(f"- Content length: {doc.get('content_length')}")

        if doc.get("description"):
            lines.append(f"- Description: {doc.get('description')}")

        lines.append("")

    return "\n".join(lines)


source_index = build_source_index(unique_docs)

source_index_path = OUTPUT_DIR / "source_index.md"

save_text(source_index_path, source_index)

print("Source index saved:", source_index_path)
print(source_index[:2000])

In [ ]:
zip_path = Path("/content/reverse_engineering_outputs.zip")

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in OUTPUT_DIR.rglob("*"):
        if file_path.is_file():
            arcname = file_path.relative_to(OUTPUT_DIR)
            zipf.write(file_path, arcname)

print("ZIP created:", zip_path)

In [ ]:
files.download(str(final_report_path))

In [ ]:
files.download(str(summary_path))

In [ ]:
files.download(str(source_index_path))

In [ ]:
files.download(str(zip_path))